# Jun-2026 LAPD cross-correlation explorer

Companion to `Jun2026_Isat_explore.ipynb`. That notebook looks at *one* fixed-bias
Isat channel's fluctuation spectrum; this one takes **two** `(scope, channel)`
pairs and asks how they relate to each other:

* **coherence** $\gamma^2(f)$ — which frequency bands the two channels share (0..1),
* **cross-phase** $\Delta\phi(f)$ — the phase lag per frequency (with a known probe
  separation this gives mode direction / speed), and
* a time-lag **cross-correlation** whose peak is the scalar time delay.

Analysis + figures live in `Jun2026_xcorr.py`; the generic DSP
(`coherence_spectrum`, `cross_phase_spectrum`, `avg_cross_spectrum`,
`cross_correlation`) lives in `data_analysis.signal`. This notebook just drives
them.

**You pick the two channels by hand** — a run has several scope groups
(`biasscope` / `lpscope` / `machscope`), so each channel is a `(scope, channel)`
pair. (Browse the channel descriptions and run setup in
`Jun2026_run_overview.ipynb` if you need them.)

Workflow: configure → positions (pick one) → read both channels at that position
→ **per-shot** figure (shot-to-shot spread) → **ensemble-averaged** figure.
*(The batch pass over many runs — `jxc.batch_xcorr()` — comes after you've
verified a few shots/positions here.)*

## 1. Configure — run file, the two channels, and the analysis window

In [11]:
import importlib
get_ipython().run_line_magic("matplotlib", "qt")
# get_ipython().run_line_magic("matplotlib", "inline")

import numpy as np
import matplotlib.pyplot as plt

import Jun2026_IV as jiv
import Jun2026_xcorr as jxc
import Jun2026_plot as jp
importlib.reload(jiv)
importlib.reload(jxc)
importlib.reload(jp)    # re-run this cell after editing Jun2026_xcorr.py / Jun2026_plot.py / Jun2026_IV.py

from data_analysis.io import open_lapd

# >>> SET THIS to the run you want to inspect <<<
ifn = r"D:\data\LAPD\jun2026-jia\07-He-800G-bias40V-Isat-p29-plane_2026-06-10.hdf5"

# >>> SET THE TWO CHANNELS to correlate, each as (scope, channel) <<<
# Same-scope pairs share the identical scope time array (no resampling); a
# cross-scope pair (e.g. one on 'biasscope') is resampled onto ch_a's grid.
ch_a = ("machscope", "C2")
ch_b = ("machscope", "C3")

# Analysis time window (ms) and Welch segment length. Larger nperseg -> finer
# frequency resolution but fewer segments per shot (more variance).
tmin_ms, tmax_ms = 1.5, 5.0
nperseg = 4096

assert ifn, "Set `ifn` to a run file first."
run = open_lapd(ifn)
print("backend:", run.backend)
print(f"ch_a = {ch_a},  ch_b = {ch_b},  window {tmin_ms}-{tmax_ms} ms,  nperseg = {nperseg}")

backend: pydaq
ch_a = ('machscope', 'C2'),  ch_b = ('machscope', 'C3'),  window 1.5-5.0 ms,  nperseg = 4096


## 2. Positions — pick one to analyze

The two channels are read only for the shots at the selected position (a
positional shot slice). For the stationary runs 00-06 every position is a repeat,
so any `pos_index` works; for a line scan pick the position of interest.

In [16]:
# If the run has more than one probe drive, set motion_group_name to the one
# carrying these channels (read_lp_positions lists the groups in its error).
motion_group_name = None    # e.g. "<Hermes>    p29_LP"

pos_array, xpos, ypos, npos, nshot = jiv.read_lp_positions(ifn, motion_group_name)

pos_index = npos//2
xcm = xpos[pos_index] if pos_index < len(xpos) else "?"
print(f"\nAnalyzing position index {pos_index} of {npos} (x={xcm} cm), {nshot} shots")

Using motion group: '<Hermes>    p29_LP'
Positions: 1369 unique (x: 37, y: 37), 5 shots/position, 6845 total shots.

Analyzing position index 684 of 1369 (x=? cm), 5 shots


## 3. Read both channels at this position

Reads the two channels' shots at the chosen position onto a common time grid,
clipped to `[tmin_ms, tmax_ms]`. Same-scope pairs use the identical scope time
array directly; a cross-scope pair resamples `ch_b` onto `ch_a`'s grid.

In [19]:
stack_a, stack_b, dt = jxc._read_pair_at_position(
    run, ch_a, ch_b, npos, nshot, pos_index, tmin_ms, tmax_ms)
print(f"stack_a: {stack_a.shape}   stack_b: {stack_b.shape}   dt = {dt:.3e} s "
      f"(fs = {1/dt/1e6:.2f} MHz)")

# Quick look at the two windowed traces (first shot) so you can see what's being
# correlated.
t_ms = (np.arange(stack_a.shape[1]) * dt + tmin_ms * 1e-3) * 1e3
fig, ax = plt.subplots()
ax.plot(t_ms, stack_a[0], lw=0.6, label=f"{ch_a[0]}/{ch_a[1]}")
ax.plot(t_ms, stack_b[0], lw=0.6, label=f"{ch_b[0]}/{ch_b[1]}", alpha=0.8)
ax.set_xlabel("t [ms]"); ax.set_ylabel("signal [V]")
ax.set_title(f"Windowed traces (shot 0) — pos {pos_index}")
ax.legend(loc="upper right"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

stack_a: (5, 17500)   stack_b: (5, 17500)   dt = 2.000e-07 s (fs = 5.00 MHz)


## 4. Per-shot figure — every shot overlaid

Runs the three methods **independently on each shot** at this position and
overlays them (one faint line per shot) so the shot-to-shot scatter is visible.
Three panels: coherence $\gamma^2(f)$, cross-phase $\Delta\phi(f)$ (only shown
where $\gamma^2 \ge$ `gamma2_floor` — phase is meaningless where the channels are
incoherent), and the time-lag cross-correlation.

`fmax_khz` clips the frequency panels to the low-frequency turbulence band.

In [20]:
fmax_khz     = 80.0    # frequency panels x-limit (kHz)
gamma2_floor = 0.5     # only draw cross-phase where coherence >= this

per_shot = jxc.xcorr_per_shot(stack_a, stack_b, dt, nperseg=nperseg)
print("per-shot gamma2:", per_shot["gamma2"].shape,
      " phase:", per_shot["phase"].shape,
      " xcorr:", per_shot["xcorr"].shape)

title = (f"pos {pos_index} (x={xcm} cm), {per_shot['gamma2'].shape[0]} shots")
jp.plot_xcorr_per_shot(per_shot, title=title, fmax_khz=fmax_khz,
                       gamma2_floor=gamma2_floor, save_fig=False, show=True)

per-shot gamma2: (5, 2049)  phase: (5, 2049)  xcorr: (5, 34999)


## 5. Ensemble-averaged figure

The trustworthy estimate: the cross-spectrum and auto-spectra are averaged over
**all shots** at this position *before* forming the coherence ratio (a single
segment gives $\gamma^2 \equiv 1$ trivially, so the average is what reveals real
shared structure). The cross-correlation panel here is computed on the two
shot-averaged traces, with its peak lag marked.

In [21]:
avg = jxc.xcorr_averaged(stack_a, stack_b, dt, nperseg=nperseg)
print("averaged over", avg["n_used"], "shots")

# Peak-lag readout (scalar time delay between the channels).
k = int(np.argmax(avg["xcorr"]))
print(f"cross-correlation peak at lag = {avg['lags'][k]*1e6:+.2f} us "
      f"(value {avg['xcorr'][k]:.3f})")

title = (f"Ensemble avg: {ch_a[0]}/{ch_a[1]} vs {ch_b[0]}/{ch_b[1]} — "
         f"pos {pos_index} (x={xcm} cm), {avg['n_used']} shots")
jp.plot_xcorr_averaged(avg, title=title, fmax_khz=fmax_khz,
                       gamma2_floor=gamma2_floor, save_fig=False, show=True)

averaged over 5 shots
cross-correlation peak at lag = +266.60 us (value 0.547)


## 6. Batch pass over many runs (run once you're satisfied)

Sections 2-5 explore one run at one position. Once the channels / window look
right, the batch routine `jxc.batch_xcorr()` does the ensemble coherence +
cross-phase for **every** run matched by `RUN_GLOB` and writes one npz
(`OUT_NPZ`) into the data dir — same schema as `Jun2026_Isat.batch_fft`. The
channels / window / `nperseg` come from the constants at the top of
`Jun2026_xcorr.py` (`CH_A`, `CH_B`, `TMIN_MS`, `TMAX_MS`, `NPERSEG`); edit them
there (or pass overrides) and re-run.

`Jun2026_plot.plot_xcorr_run(ifn)` then reloads a run's entry from that npz and
draws the coherence + cross-phase figure.

In [ ]:
# >>> Uncomment to run the batch pass on THIS run (progress bar ticks per shot) <<<
# Averages coherence/cross-phase over every shot and writes the result into the
# run's co-located npz (<run_num>-xcorr-data.npz, next to the HDF5), keyed by the
# channel pair so more pairs can be added to the same file later.
# out_path = jxc.batch_xcorr(ifn, ch_a=ch_a, ch_b=ch_b,
#                            tmin_ms=tmin_ms, tmax_ms=tmax_ms, nperseg=nperseg)
# print("wrote", out_path)

# Then reload + plot this pair from the run's npz:
# jp.plot_xcorr_run(ifn, ch_a=ch_a, ch_b=ch_b, save_fig=True, show=True, fmax_khz=fmax_khz)